# Waymaker Artemis — Gemma 4 E4B CUDA 재현 노트북 (Mission 005-C.1)

무료 Colab T4에서 **저장소 코드**로 공식 `google/gemma-4-E4B-it`(revision `ee0ef60…`)를 NF4 4비트로 올리고,
합성 시나리오 3개(근거 없음, 가상 규칙 1건, 상충 규칙 2건)만 실행합니다.

- **코드 고정**: 커밋 `59858c507ca48e8429b31baebbd96ca9a5ef5cda`를 받습니다. 브랜치 끝을 따라가지 않습니다.
- **패키지 고정**: transformers 5.20.2, bitsandbytes 0.50.2, Colab 기본 torch 2.11.0+cu130.
  버전이 다르면 실행을 막습니다. 다른 버전으로 돌린 결과는 재현이 아닙니다.
- **비용**: 무료 런타임만 씁니다. 유료 GPU, 컴퓨팅 단위 구매, Google Cloud 결제는 쓰지 않습니다.
- **비밀값**: 토큰이 필요 없습니다. 노트북에 토큰을 적지 마세요.
- **해석 범위**: 실행과 응답 형식만 봅니다. 법률 정확도와 Official Gold Accuracy는 `N/A`입니다.
- **상태**: 이 파일은 아직 실행하지 않았습니다(`not_reexecuted`). 2026-10-30 원본 실행 출력을 이 노트북 결과로 쓰지 않습니다.

In [ ]:
# 1. GPU·VRAM·디스크 확인 (읽기 전용)
import shutil, subprocess, sys
print(sys.version)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
print('Disk free GB:', round(shutil.disk_usage('/content').free / 1e9, 2))

## 2. 무료 실행 환경 확인

코드로는 결제 상태를 알 수 없습니다. 실행 전에 직접 확인하세요.

1. 메뉴 **런타임 → 런타임 유형 변경**에서 `T4 GPU`인지 봅니다. A100·L4 같은 유료 가속기는 고르지 않습니다.
2. 오른쪽 위 **리소스 보기**에서 컴퓨팅 단위를 쓰지 않는 무료 상태인지 봅니다.
3. 둘 다 맞으면 아래 셀의 값을 `True`로 바꿉니다.

In [ ]:
FREE_TIER_CONFIRMED = False  # 위 두 가지를 직접 확인한 뒤에만 True로 바꿉니다
assert FREE_TIER_CONFIRMED, '무료 T4 런타임인지 먼저 확인하세요'
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
assert gpu == 'Tesla T4', f'예상한 무료 T4가 아닙니다: {gpu!r}'

In [ ]:
# 3. 코드와 패키지 버전 고정
CODE_COMMIT = '59858c507ca48e8429b31baebbd96ca9a5ef5cda'
!git init -q /content/Visable
!git -C /content/Visable remote add origin https://github.com/club-paradiso/Visable.git
!git -C /content/Visable sparse-checkout set experiments/waymaker-ai
!git -C /content/Visable fetch -q --depth 1 --filter=blob:none origin {CODE_COMMIT}
!git -C /content/Visable checkout -q FETCH_HEAD
head = subprocess.run(['git', '-C', '/content/Visable', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert head == CODE_COMMIT, f'코드 커밋 불일치: {head}'
%pip install -q "transformers==5.20.2" "bitsandbytes==0.50.2"
import importlib.metadata as md
installed = {n: md.version(n) for n in ('torch', 'transformers', 'bitsandbytes', 'accelerate', 'huggingface-hub', 'tokenizers')}
print(installed)
print(subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True).stdout)
# torch는 다시 설치하지 않습니다. 다르면 다음 단계의 사전 검사가 실행을 막습니다.

In [ ]:
# 4. 체크포인트 다운로드(이 셀만 가중치를 받습니다, 약 16GB)와 SHA-256 확인
import json, os
REPO = '/content/Visable/experiments/waymaker-ai'
sys.path[:0] = [REPO, REPO + '/artemis_bootstrap']
policy = json.load(open(REPO + '/configs/artemis_cuda_runtime.json'))
MODEL, REVISION = policy['model']['checkpoint'], policy['model']['revision']
assert (MODEL, REVISION) == ('google/gemma-4-E4B-it', 'ee0ef6023621cff504d758262d4e04895a5af4a2')
need = policy['limits']['min_free_disk_gb_for_download']
assert shutil.disk_usage('/content').free / 1e9 >= need, f'디스크 여유 공간이 {need}GB보다 적습니다'
from huggingface_hub import snapshot_download
from pathlib import Path
snapshot = Path(snapshot_download(MODEL, revision=REVISION, token=False))
from waymaker_ai import cuda_policy
verified = cuda_policy.validate_snapshot(snapshot)  # 1~2분
print(verified['weight_integrity'], verified['artifact_integrity']['model.safetensors']['digest'])

In [ ]:
# 5. CUDA 사전 검사. 이후로는 네트워크를 쓰지 않습니다.
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
import gemma4_cuda_runtime as cuda
check = cuda.preflight()  # 같은 프로세스라 SHA-256 결과를 다시 계산하지 않습니다
print('blockers:', check['blockers'])
print('version drift:', check['probe']['version_drift'])

In [ ]:
# 6~7. 실제 모델 로딩과 NF4 검증 (올라간 모듈을 직접 읽습니다)
prepared = cuda.PreparedRun()
loaded = prepared.prepare()
print('loaded:', loaded, '| blocker:', prepared.blocker)
quant = prepared.header.get('load', {}).get('quantization', {})
print({k: quant.get(k) for k in ('passed', 'quant_types', 'double_quant', 'compute_dtypes',
                                 'parameter_devices', 'quantized_decoder_layers', 'linear4bit_modules')})
print('load seconds:', prepared.header.get('load', {}).get('load_seconds'))

In [ ]:
# 8~9. 합성 시나리오 3개 실행, 원본 출력과 메트릭 저장. 기록을 덮어쓰지 않습니다.
# 종료 코드 0=completed, 2=generated_but_invalid, 1=failed, 3=not_executed, 4=기록 자체 검사 실패
import datetime, probe_gemma4_cuda
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT = f'/content/artemis_cuda_record_{stamp}.json'
exit_code = probe_gemma4_cuda.main(['--out', OUT, '--max-new-tokens', '96'], prepared=prepared)
record = json.load(open(OUT))
print('exit:', exit_code, '| status:', record['execution_status'], '| record check:', record['record_validation'])
for case in record['cases']:
    print(case['case_id'], case['execution_status'], case.get('generated_token_count'),
          case.get('finish_reason'), repr(case.get('raw_output')))

In [ ]:
# 10. 결과 파일 내려받기
from google.colab import files
files.download(OUT)

## 끝난 뒤

- 내려받은 JSON을 그대로 `experiments/waymaker-ai/artemis_bootstrap/runs/` 아래 새 폴더에 넣습니다. 내용을 고치지 않습니다.
- 메뉴 **런타임 → 런타임 연결 해제 및 삭제**로 GPU를 반납합니다. 무료 사용량을 아끼고 디스크의 모델 사본도 지워집니다.